In [0]:
%python
dbutils.widgets.text('env','dev')
env = dbutils.widgets.get('env')
env

In [0]:
%sql
-- =============================================================================
-- 00_setup_tables.sql
-- DDL for Bronze / Silver / Gold / Audit tables.
-- All tables use Delta format with liquid clustering or Z-ORDER hints.
-- Replace <env> before executing.
-- =============================================================================

-- ─────────────────────────────────────────────────────────────────────────────
-- AUDIT SCHEMA
-- ─────────────────────────────────────────────────────────────────────────────

CREATE TABLE IF NOT EXISTS fin_platform_${env}.audit.pipeline_runs (
  run_id          STRING    NOT NULL COMMENT 'UUID for this pipeline execution',
  pipeline_name   STRING    NOT NULL,
  layer           STRING    NOT NULL COMMENT 'bronze | silver | gold',
  env             STRING    NOT NULL,
  status          STRING    NOT NULL COMMENT 'RUNNING | SUCCESS | FAILED',
  started_at      TIMESTAMP NOT NULL,
  completed_at    TIMESTAMP,
  rows_read       LONG,
  rows_written    LONG,
  source_files    STRING    COMMENT 'Comma-separated list of ingested files',
  error_message   STRING,
  spark_app_id    STRING
)
USING DELTA
COMMENT 'One row per notebook/pipeline execution'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true'
);

CREATE TABLE IF NOT EXISTS fin_platform_${env}.audit.dq_results (
  pipeline    STRING,
  layer       STRING,
  table_name  STRING,
  run_date    STRING,
  rule_name   STRING,
  severity    STRING,
  passed      BOOLEAN,
  details     STRING
)
USING DELTA
COMMENT 'Data quality check results per pipeline run'
TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true');

-- ─────────────────────────────────────────────────────────────────────────────
-- BRONZE SCHEMA  – raw, append-only, source-faithful
-- ─────────────────────────────────────────────────────────────────────────────

CREATE TABLE IF NOT EXISTS fin_platform_${env}.bronze.trade_executions (
  -- source columns (all STRING to preserve fidelity)
  trade_id          STRING,
  trade_date        STRING,
  settlement_date   STRING,
  instrument_id     STRING,
  instrument_code   STRING,
  asset_class       STRING,
  counterparty_id   STRING,
  trader_id         STRING,
  portfolio_id      STRING,
  account_id        STRING,
  side              STRING,
  quantity          STRING,
  execution_price   STRING,
  gross_value       STRING,
  commission        STRING,
  currency          STRING,
  venue             STRING,
  trade_status      STRING,
  source_system     STRING,
  load_date         STRING,
  file_date         STRING,
  -- pipeline metadata
  _ingest_timestamp TIMESTAMP NOT NULL,
  _source_file      STRING,
  _pipeline_run_id  STRING,
  _row_hash         STRING    COMMENT 'MD5 of all source fields for dedup'
)
USING DELTA
PARTITIONED BY (file_date)
COMMENT 'Raw trade executions – bronze layer, append-only'
TBLPROPERTIES (
  'delta.enableChangeDataFeed'       = 'false',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'quality'                          = 'bronze'
);

CREATE TABLE IF NOT EXISTS fin_platform_${env}.bronze.market_prices (
  price_id          STRING,
  price_date        STRING,
  instrument_id     STRING,
  instrument_code   STRING,
  asset_class       STRING,
  open_price        STRING,
  high_price        STRING,
  low_price         STRING,
  close_price       STRING,
  prev_close        STRING,
  volume            STRING,
  vwap              STRING,
  currency          STRING,
  price_source      STRING,
  source_system     STRING,
  load_date         STRING,
  file_date         STRING,
  -- pipeline metadata
  _ingest_timestamp TIMESTAMP NOT NULL,
  _source_file      STRING,
  _pipeline_run_id  STRING,
  _row_hash         STRING
)
USING DELTA
PARTITIONED BY (file_date)
COMMENT 'Raw market prices – bronze layer, append-only'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'quality'                          = 'bronze'
);

-- ─────────────────────────────────────────────────────────────────────────────
-- SILVER SCHEMA  – cleansed, typed, deduplicated
-- ─────────────────────────────────────────────────────────────────────────────

CREATE TABLE IF NOT EXISTS fin_platform_${env}.silver.trade_executions (
  trade_id          STRING        NOT NULL,
  trade_date        DATE          NOT NULL,
  settlement_date   DATE,
  instrument_id     STRING        NOT NULL,
  instrument_code   STRING        NOT NULL,
  asset_class       STRING        NOT NULL,
  counterparty_id   STRING,
  trader_id         STRING,
  portfolio_id      STRING,
  account_id        STRING,
  side              STRING        NOT NULL   COMMENT 'BUY | SELL',
  quantity          LONG          NOT NULL,
  execution_price   DECIMAL(18,6) NOT NULL,
  gross_value       DECIMAL(18,2) NOT NULL,
  commission        DECIMAL(18,2),
  net_value         DECIMAL(18,2) COMMENT 'gross_value + commission (signed)',
  currency          STRING        NOT NULL,
  venue             STRING,
  trade_status      STRING,
  source_system     STRING,
  -- pipeline metadata
  _ingest_timestamp TIMESTAMP,
  _pipeline_run_id  STRING,
  _silver_processed_at TIMESTAMP,
  _row_hash         STRING
)
USING DELTA
PARTITIONED BY (trade_date)
COMMENT 'Cleansed trade executions – silver layer'
TBLPROPERTIES (
  'delta.enableChangeDataFeed'       = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'quality'                          = 'silver'
);

CREATE TABLE IF NOT EXISTS fin_platform_${env}.silver.market_prices (
  price_id          STRING        NOT NULL,
  price_date        DATE          NOT NULL,
  instrument_id     STRING        NOT NULL,
  instrument_code   STRING        NOT NULL,
  asset_class       STRING        NOT NULL,
  open_price        DECIMAL(18,6),
  high_price        DECIMAL(18,6),
  low_price         DECIMAL(18,6),
  close_price       DECIMAL(18,6) NOT NULL,
  prev_close        DECIMAL(18,6),
  daily_return_pct  DECIMAL(10,6) COMMENT '(close - prev_close) / prev_close * 100',
  intraday_range    DECIMAL(18,6) COMMENT 'high - low',
  volume            LONG,
  vwap              DECIMAL(18,6),
  currency          STRING,
  price_source      STRING,
  source_system     STRING,
  -- pipeline metadata
  _ingest_timestamp TIMESTAMP,
  _pipeline_run_id  STRING,
  _silver_processed_at TIMESTAMP,
  _row_hash         STRING
)
USING DELTA
PARTITIONED BY (price_date)
COMMENT 'Cleansed market prices – silver layer'
TBLPROPERTIES (
  'delta.enableChangeDataFeed'       = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'quality'                          = 'silver'
);

-- ─────────────────────────────────────────────────────────────────────────────
-- GOLD SCHEMA  – business-level aggregates
-- ─────────────────────────────────────────────────────────────────────────────

CREATE TABLE IF NOT EXISTS fin_platform_${env}.gold.daily_trade_summary (
  trade_date          DATE     NOT NULL,
  asset_class         STRING   NOT NULL,
  instrument_code     STRING   NOT NULL,
  currency            STRING   NOT NULL,
  buy_count           LONG,
  sell_count          LONG,
  total_trade_count   LONG,
  total_buy_quantity  LONG,
  total_sell_quantity LONG,
  total_buy_value     DECIMAL(24,2),
  total_sell_value    DECIMAL(24,2),
  net_value           DECIMAL(24,2)  COMMENT 'buy_value - sell_value',
  avg_execution_price DECIMAL(18,6),
  total_commission    DECIMAL(18,2),
  unique_traders      LONG,
  unique_portfolios   LONG,
  unique_counterparties LONG,
  -- pipeline metadata
  _pipeline_run_id    STRING,
  _gold_processed_at  TIMESTAMP
)
USING DELTA
PARTITIONED BY (trade_date)
COMMENT 'Daily trade summary by instrument – gold layer'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'quality'                          = 'gold'
);

CREATE TABLE IF NOT EXISTS fin_platform_${env}.gold.portfolio_daily_pnl (
  trade_date          DATE          NOT NULL,
  portfolio_id        STRING        NOT NULL,
  instrument_code     STRING        NOT NULL,
  asset_class         STRING        NOT NULL,
  currency            STRING        NOT NULL,
  net_quantity        LONG          COMMENT 'buy_qty - sell_qty',
  avg_cost            DECIMAL(18,6),
  close_price         DECIMAL(18,6),
  market_value        DECIMAL(24,2) COMMENT 'net_quantity * close_price',
  realised_pnl        DECIMAL(24,2),
  unrealised_pnl      DECIMAL(24,2),
  total_pnl           DECIMAL(24,2),
  -- pipeline metadata
  _pipeline_run_id    STRING,
  _gold_processed_at  TIMESTAMP
)
USING DELTA
PARTITIONED BY (trade_date)
COMMENT 'Portfolio-level daily P&L – gold layer'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'quality'                          = 'gold'
);

CREATE TABLE IF NOT EXISTS fin_platform_${env}.gold.instrument_market_snapshot (
  price_date          DATE          NOT NULL,
  instrument_code     STRING        NOT NULL,
  asset_class         STRING        NOT NULL,
  currency            STRING        NOT NULL,
  close_price         DECIMAL(18,6),
  vwap                DECIMAL(18,6),
  volume              LONG,
  daily_return_pct    DECIMAL(10,6),
  intraday_range      DECIMAL(18,6),
  high_52w            DECIMAL(18,6) COMMENT 'Rolling 52-week high (computed at gold)',
  low_52w             DECIMAL(18,6),
  price_source        STRING,
  -- pipeline metadata
  _pipeline_run_id    STRING,
  _gold_processed_at  TIMESTAMP
)
USING DELTA
PARTITIONED BY (price_date)
COMMENT 'Daily instrument market snapshot enriched with rolling stats – gold layer'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'quality'                          = 'gold'
);
